# GroupDNA - WhatsApp Group Chat Analyzer
**Name:** Parth Sharma  
**Roll Number:** PARTH1939160  
**Batch:** Batch 73  
**Date:** 24 November 2026


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [24]:
import numpy as np
from datetime import datetime, timedelta

FILE_NAME = "hostel_bois.txt"
GROUP_NAME = "Hostel Bois 4ever"

def read_chat_file(filename):
    for path in [filename, "/content/" + filename]:
        try:
            with open(path, "r", encoding="utf-8") as f:
                return f.read().split("\n")
        except FileNotFoundError:
            continue
    raise FileNotFoundError(f"File not found: {filename}")


### Feature 1: Chat Parser


In [25]:
def starts_with_date(line):
    if len(line) < 8:
        return False
    comma_idx = line.find(",")
    if comma_idx == -1 or comma_idx > 10:
        return False
    parts = line[:comma_idx].strip().split("/")
    if len(parts) != 3:
        return False
    d, m, y = parts
    return d.isdigit() and m.isdigit() and y.isdigit() and len(y) in (2, 4)


def parse_dt(ts_str):
    cleaned = ts_str.replace("\u202f", " ").strip()
    for fmt in ("%d/%m/%y, %H:%M", "%d/%m/%Y, %H:%M", "%d/%m/%y, %I:%M %p", "%d/%m/%Y, %I:%M %p"):
        try:
            return datetime.strptime(cleaned, fmt)
        except ValueError:
            continue
    return None


def parse_chat(lines):
    messages = []
    system_msgs = 0
    media_msgs = 0
    deleted_msgs = 0
    media_by_user = {}
    deleted_by_user = {}

    for raw in lines:
        line = raw.strip()
        if not line:
            continue

        # Multi-line message continuation
        if not starts_with_date(line):
            if len(messages) > 0 and not messages[-1]["is_media"] and not messages[-1]["is_deleted"]:
                messages[-1]["text"] += " " + line
            continue

        if " - " in line:
            ts_str, rest = line.split(" - ", 1)
        elif " – " in line:
            ts_str, rest = line.split(" – ", 1)
        else:
            system_msgs += 1
            continue

        dt_obj = parse_dt(ts_str)
        if dt_obj is None or ":" not in rest:
            system_msgs += 1
            continue

        sender, text = rest.split(":", 1)
        sender = sender.strip()
        text = text.strip()
        if not sender:
            system_msgs += 1
            continue

        is_media = (text.lower() == "<media omitted>")
        if is_media:
            media_msgs += 1
            media_by_user[sender] = media_by_user.get(sender, 0) + 1

        is_deleted = text.lower() in ("this message was deleted", "you deleted this message")
        if is_deleted:
            deleted_msgs += 1
            deleted_by_user[sender] = deleted_by_user.get(sender, 0) + 1

        messages.append({
            "timestamp": ts_str.strip(),
            "dt": dt_obj,
            "sender": sender,
            "text": text,
            "is_media": is_media,
            "is_deleted": is_deleted,
        })

    counts = {
        "system": system_msgs,
        "media": media_msgs,
        "deleted": deleted_msgs,
        "media_by_user": media_by_user,
        "deleted_by_user": deleted_by_user,
    }
    return messages, counts


raw_lines = read_chat_file(FILE_NAME)
messages, parse_counts = parse_chat(raw_lines)

if len(messages) > 0:
    members_set = set([m["sender"] for m in messages])
    days_span = (messages[-1]["dt"].date() - messages[0]["dt"].date()).days + 1
    print(
        f"Successfully parsed {len(messages):,} messages from {len(members_set)} participants "
        f"over {days_span} days, skipped {parse_counts['system']} system messages, "
        f"{parse_counts['media']} media-omitted, {parse_counts['deleted']} deleted messages."
    )
    for m in messages[:5]:
        print(f"  [{m['timestamp']}] {m['sender']}: {m['text'][:60]}")

Successfully parsed 3,174 messages from 6 participants over 60 days, skipped 4 system messages, 32 media-omitted, 15 deleted messages.
  [01/04/24, 01:17] Rahul: scene fix
  [01/04/24, 01:17] Rahul: haan
  [01/04/24, 01:18] Rahul: kya scene
  [01/04/24, 02:13] Rahul: abhi free hai?
  [01/04/24, 02:13] Rahul: abey


### Feature 2: Group Overview


In [26]:
def get_overview(messages):
    if len(messages) == 0:
        return None

    start_dt = messages[0]["dt"]
    end_dt = messages[-1]["dt"]
    total_days = (end_dt.date() - start_dt.date()).days + 1

    msg_count = {}
    text_count = {}
    word_count = {}

    for m in messages:
        s = m["sender"]
        msg_count[s] = msg_count.get(s, 0) + 1
        if not m["is_media"] and not m["is_deleted"]:
            text_count[s] = text_count.get(s, 0) + 1
            word_count[s] = word_count.get(s, 0) + len(m["text"].split())

    ranked = sorted(msg_count.items(), key=lambda x: x[1], reverse=True)

    avg_words = {}
    for s in msg_count:
        tc = text_count.get(s, 0)
        avg_words[s] = (word_count.get(s, 0) / tc) if tc > 0 else 0.0

    return {
        "total_messages": len(messages),
        "start_str": start_dt.strftime("%d %B %Y"),
        "end_str": end_dt.strftime("%d %B %Y"),
        "start_date": start_dt.date(),
        "end_date": end_dt.date(),
        "total_days": total_days,
        "participants": [name for name, _ in ranked],
        "msg_count": msg_count,
        "ranked": ranked,
        "avg_words": avg_words,
    }


overview = get_overview(messages)

if overview:
    print("=" * 55)
    print("GROUP OVERVIEW")
    print("=" * 55)
    print(f"Group          : {GROUP_NAME}")
    print(f"Period         : {overview['start_str']} to {overview['end_str']} ({overview['total_days']} days)")
    print(f"Total messages : {overview['total_messages']:,}")
    print(f"Participants   : {len(overview['participants'])}")
    print("\nMESSAGES PER PERSON")
    for name, cnt in overview["ranked"]:
        pct = (cnt / overview["total_messages"]) * 100
        print(f"  {name:<10} : {cnt:>5} ({pct:>5.1f}%)  | avg {overview['avg_words'][name]:>4.1f} words/msg")


GROUP OVERVIEW
Group          : Hostel Bois 4ever
Period         : 01 April 2024 to 30 May 2024 (60 days)
Total messages : 3,174
Participants   : 6

MESSAGES PER PERSON
  Rahul      :   953 ( 30.0%)  | avg  2.6 words/msg
  Priya      :   718 ( 22.6%)  | avg  5.0 words/msg
  Neha       :   635 ( 20.0%)  | avg  5.3 words/msg
  Aman       :   490 ( 15.4%)  | avg  5.0 words/msg
  Karan      :   354 ( 11.2%)  | avg 57.0 words/msg
  Vikas      :    24 (  0.8%)  | avg  1.8 words/msg


### Feature 3: Most Active Day and Hour


In [27]:
def get_most_active(messages, total_days):
    if len(messages) == 0:
        return None

    by_day = {}
    by_hour = {}

    for m in messages:
        d_key = m["dt"].strftime("%d %B %Y")
        h_key = m["dt"].hour
        by_day[d_key] = by_day.get(d_key, 0) + 1
        by_hour[h_key] = by_hour.get(h_key, 0) + 1

    top_day = max(by_day, key=lambda d: by_day[d])
    top_hour = max(by_hour, key=lambda h: by_hour[h])

    return {
        "busiest_day": top_day,
        "busiest_day_msgs": by_day[top_day],
        "busiest_hour": top_hour,
        "busiest_hour_msgs": by_hour[top_hour],
        "busiest_hour_avg": by_hour[top_hour] / max(total_days, 1),
    }


peaks = get_most_active(messages, overview["total_days"]) if overview else None

if peaks:
    hr = peaks["busiest_hour"]
    print(f"Busiest day  : {peaks['busiest_day']} ({peaks['busiest_day_msgs']} messages)")
    print(f"Busiest hour : {hr:02d}:00 - {(hr + 1) % 24:02d}:00 (avg {peaks['busiest_hour_avg']:.1f} messages/day)")


Busiest day  : 04 May 2024 (76 messages)
Busiest hour : 18:00 - 19:00 (avg 4.1 messages/day)


### Feature 4: Activity Heatmap (NumPy)


In [28]:
def make_heatmap(messages, participants):
    matrix = np.zeros((len(participants), 24), dtype=int)
    row_map = {name: i for i, name in enumerate(participants)}

    for m in messages:
        matrix[row_map[m["sender"]], m["dt"].hour] += 1
    return matrix


def night_msg_percent(matrix, idx):
    total = int(np.sum(matrix[idx]))
    if total == 0:
        return 0.0
    # Hours 23:00 to 04:59
    night = int(matrix[idx, 23] + np.sum(matrix[idx, 0:5]))
    return (night / total) * 100.0


def format_heatmap(matrix, participants):
    output = ["  " + " " * 10 + "00    03    06    09    12    15    18    21"]
    row_maxes = np.max(matrix, axis=1)

    for i, name in enumerate(participants):
        r_max = int(row_maxes[i])
        blocks = []
        for h in range(24):
            val = int(matrix[i, h])
            if r_max == 0:
                ch = "."
            else:
                ratio = val / r_max
                if ratio <= 0.25:
                    ch = "."
                elif ratio <= 0.50:
                    ch = "░"
                elif ratio <= 0.75:
                    ch = "▒"
                else:
                    ch = "█"
            blocks.append(ch)

        tag = "  <- NIGHT OWL" if night_msg_percent(matrix, i) > 60 else ""
        output.append(f"  {name:<10}{' '.join(blocks)}{tag}")

    output.append("  Legend: . (0-25%)  ░ (25-50%)  ▒ (50-75%)  █ (75-100%)")
    return output


if overview:
    participants = overview["participants"]
    heatmap = make_heatmap(messages, participants)
    print("ACTIVITY HEATMAP (messages by hour)")
    for row_line in format_heatmap(heatmap, participants):
        print(row_line)


ACTIVITY HEATMAP (messages by hour)
            00    03    06    09    12    15    18    21
  Rahul     . . . . . . . . . . . . ▒ ░ ░ ▒ ▒ ░ █ ▒ ░ █ ▒ ▒
  Priya     . . . . . . . ░ ▒ █ █ █ █ ▒ ▒ ░ ░ ▒ ▒ █ ▒ ░ ░ .
  Neha      . . . . . ░ . . ▒ █ █ ░ ▒ ▒ ░ . ▒ █ █ █ ▒ ░ ░ ░
  Aman      ▒ █ ▒ ▒ █ . . . . . . . . . . . . . . . . . . ▒  <- NIGHT OWL
  Karan     . . . . . . . . ░ ░ ▒ ░ █ ▒ █ ▒ ▒ ▒ ▒ █ ▒ ░ . .
  Vikas     . . . . . . . ░ █ ░ ░ . ▒ ▒ . ░ ░ █ ▒ ▒ ░ ░ ░ ▒
  Legend: . (0-25%)  ░ (25-50%)  ▒ (50-75%)  █ (75-100%)


### Feature 5: Top Words


In [29]:
STOP_WORDS = {
    "i", "is", "the", "a", "an", "and", "or", "to", "of", "in", "on", "for",
    "it", "this", "that", "you", "me", "my", "we", "are", "be", "at", "so",
    "do", "not", "no", "but", "with", "just", "was", "have", "has", "had",
    "can", "will", "am", "im", "i'm", "its", "it's", "from", "by", "as",
    "if", "up", "out", "about", "all", "too", "very", "your", "our", "they",
    "he", "she", "him", "her", "them", "what", "when", "where", "how", "who",
    "why", "get", "got", "one", "like", "dont", "don't", "cant", "can't",
    "u", "ur", "also", "then", "than", "here", "there", "been", "were",
    "would", "could", "should", "now", "more", "some", "only", "still"
}

PUNCT = ".,!?;:\"'()[]{}<>-_/\\*~`@#$%^&+="


def get_top_words(messages):
    group_freq = {}
    user_freq = {}

    for m in messages:
        if m["is_media"] or m["is_deleted"]:
            continue

        s = m["sender"]
        if s not in user_freq:
            user_freq[s] = {}

        for word in m["text"].split():
            w = word.lower().strip(PUNCT)
            if len(w) <= 1 or w in STOP_WORDS or w.isdigit():
                continue
            group_freq[w] = group_freq.get(w, 0) + 1
            user_freq[s][w] = user_freq[s].get(w, 0) + 1

    top_10 = sorted(group_freq.items(), key=lambda x: x[1], reverse=True)[:10]
    top_5_each = {
        s: sorted(d.items(), key=lambda x: x[1], reverse=True)[:5]
        for s, d in user_freq.items()
    }
    return top_10, top_5_each


top_10_words, top_5_per_person = get_top_words(messages)

print("THIS GROUP'S FAVOURITE WORDS")
if len(top_10_words) > 0:
    max_cnt = top_10_words[0][1]
    for w, cnt in top_10_words:
        bar = "█" * max(1, int((cnt / max_cnt) * 28))
        print(f"  {w:<12} {bar:<28} {cnt}")

print("\nTOP 5 WORDS PER PERSON")
for s, wlist in top_5_per_person.items():
    print(f"  {s:<10} : " + ", ".join([f"{w} ({c})" for w, c in wlist]))


THIS GROUP'S FAVOURITE WORDS
  guys         ████████████████████████████ 318
  hai          ███████████████████████      268
  today        ██████████████████████       257
  his          ███████████████████          217
  which        █████████████████            202
  everyone     ████████████████             187
  telling      ███████████████              179
  bhai         ██████████████               160
  started      █████████████                150
  scene        ████████████                 145

TOP 5 WORDS PER PERSON
  Rahul      : hai (263), bhai (159), scene (144), kya (133), yaar (105)
  Priya      : please (141), everyone (137), aman (93), anyone (90), okay (80)
  Karan      : his (217), which (202), telling (179), today (171), started (150)
  Neha       : guys (101), ok (52), today (48), way (48), wait (48)
  Aman       : sleep (71), anyone (49), wonder (43), night (41), does (33)
  Vikas      : hai (5), haha (4), sorry (3), busy (3), tha (3)


### Feature 6: Response Speed and Silent Streaks


In [30]:
def fmt_time(sec):
    if sec is None:
        return "N/A"
    if sec < 60:
        return f"{sec:.0f} seconds"
    if sec < 3600:
        return f"{sec / 60:.1f} minutes"
    return f"{sec / 3600:.1f} hours"


def get_response_stats(messages, participants):
    gaps = {p: [] for p in participants}

    for i in range(1, len(messages)):
        prev_m = messages[i - 1]
        curr_m = messages[i]
        if curr_m["sender"] != prev_m["sender"]:
            diff = (curr_m["dt"] - prev_m["dt"]).total_seconds()
            if diff >= 0:
                gaps[curr_m["sender"]].append(diff)

    avgs = {}
    for p in participants:
        avgs[p] = (sum(gaps[p]) / len(gaps[p])) if len(gaps[p]) > 0 else None

    valid = [(p, t) for p, t in avgs.items() if t is not None]
    fastest = min(valid, key=lambda x: x[1]) if valid else None
    slowest = max(valid, key=lambda x: x[1]) if valid else None
    return avgs, fastest, slowest


def get_silent_streaks(messages, participants, start_date, end_date):
    active = {p: set() for p in participants}
    for m in messages:
        active[m["sender"]].add(m["dt"].date())

    all_days = []
    curr = start_date
    while curr <= end_date:
        all_days.append(curr)
        curr += timedelta(days=1)

    streaks = {}
    for p in participants:
        best_len = 0
        best_start = None
        best_end = None
        curr_len = 0
        curr_start = None
        silent_total = 0

        for d in all_days:
            if d not in active[p]:
                silent_total += 1
                if curr_len == 0:
                    curr_start = d
                curr_len += 1
                if curr_len > best_len:
                    best_len = curr_len
                    best_start = curr_start
                    best_end = d
            else:
                curr_len = 0
                curr_start = None

        streaks[p] = {
            "longest": best_len,
            "start": best_start,
            "end": best_end,
            "silent_days": silent_total,
            "total_days": len(all_days),
        }
    return streaks


if overview:
    resp_avgs, fastest, slowest = get_response_stats(messages, participants)
    streaks = get_silent_streaks(messages, participants, overview["start_date"], overview["end_date"])

    print("RESPONSE PATTERNS")
    if fastest and slowest:
        print(f"  Fastest replier : {fastest[0]} (avg {fmt_time(fastest[1])})")
        print(f"  Slowest replier : {slowest[0]} (avg {fmt_time(slowest[1])})")

    print("\nLONGEST SILENT STREAKS")
    for name, info in sorted(streaks.items(), key=lambda x: x[1]["longest"], reverse=True):
        d = info["longest"]
        if d == 0:
            print(f"  {name:<10} : 0 days (never went silent)")
        elif d == 1:
            print(f"  {name:<10} : 1 day ({info['start'].strftime('%d %b')})")
        else:
            print(f"  {name:<10} : {d} days ({info['start'].strftime('%d %b')} - {info['end'].strftime('%d %b')})")


RESPONSE PATTERNS
  Fastest replier : Rahul (avg 34.9 minutes)
  Slowest replier : Aman (avg 55.4 minutes)

LONGEST SILENT STREAKS
  Vikas      : 11 days (23 Apr - 03 May)
  Rahul      : 0 days (never went silent)
  Priya      : 0 days (never went silent)
  Neha       : 0 days (never went silent)
  Aman       : 0 days (never went silent)
  Karan      : 0 days (never went silent)


### Feature 7: Personality Archetypes


In [31]:
CARING_WORDS = [
    "okay", "safe", "eat", "sleep", "take care",
    "are you", "please", "reminder", "drink water", "don't forget"
]
LAUGH_WORDS = ["lol", "lmao", "haha", "rofl", "lmfao"]


def user_texts(person, messages):
    return [m["text"] for m in messages if m["sender"] == person and not m["is_media"] and not m["is_deleted"]]


def score_spammer(person, messages):
    bursts = []
    run = 0
    for m in messages:
        if m["sender"] == person:
            run += 1
        elif run > 0:
            bursts.append(run)
            run = 0
    if run > 0:
        bursts.append(run)

    avg_b = sum(bursts) / len(bursts) if bursts else 0.0
    sc = (avg_b / 3.0) if avg_b > 3.0 else (avg_b / 3.0) * 0.4
    return sc, f"avg {avg_b:.1f} msgs in a row"


def score_group_mom(person, messages):
    texts = user_texts(person, messages)
    if not texts:
        return 0.0, "caring keyword score: 0"
    hits = 0
    caring_msgs = 0
    for t in texts:
        low = t.lower()
        found = False
        for kw in CARING_WORDS:
            if kw in low:
                hits += 1
                found = True
        if found:
            caring_msgs += 1
    pct = (caring_msgs / len(texts)) * 100.0
    sc = (pct / 18.0) if pct > 18.0 else (pct / 18.0) * 0.4
    return sc, f"caring keyword score: {hits} ({pct:.1f}% msgs)"


def score_night_owl(idx, matrix):
    pct = night_msg_percent(matrix, idx)
    sc = (pct / 60.0) if pct > 60.0 else (pct / 60.0) * 0.4
    return sc, f"{pct:.1f}% msgs between 23h-04h"


def score_storyteller(person, messages):
    texts = user_texts(person, messages)
    if not texts:
        return 0.0, "avg 0.0 words per msg"
    avg_w = sum([len(t.split()) for t in texts]) / len(texts)
    sc = (avg_w / 30.0) if avg_w > 30.0 else (avg_w / 30.0) * 0.4
    return sc, f"avg {avg_w:.1f} words per msg"


def score_drama_queen(person, messages):
    texts = user_texts(person, messages)
    if not texts:
        return 0.0, "0.0% ALL-CAPS / dramatic msgs"
    cnt = 0
    for t in texts:
        if (len(t) >= 3 and t.isupper()) or t.count("!") >= 2:
            cnt += 1
    pct = (cnt / len(texts)) * 100.0
    sc = (pct / 30.0) if pct > 30.0 else (pct / 30.0) * 0.4
    return sc, f"{pct:.1f}% ALL-CAPS / dramatic msgs"


def score_ghost(person, streaks):
    info = streaks[person]
    pct = (info["silent_days"] / max(info["total_days"], 1)) * 100.0
    sc = (pct / 60.0) if pct > 60.0 else (pct / 60.0) * 0.4
    return sc, f"silent on {info['silent_days']} of {info['total_days']} days ({pct:.0f}%)"


def score_comedian(person, messages):
    texts = user_texts(person, messages)
    if not texts:
        return 0.0, "0.0% laughing msgs"
    cnt = 0
    for t in texts:
        low = t.lower()
        if any(lw in low for lw in LAUGH_WORDS):
            cnt += 1
    pct = (cnt / len(texts)) * 100.0
    sc = (pct / 20.0) if pct > 20.0 else (pct / 20.0) * 0.4
    return sc, f"{pct:.1f}% laughing msgs"


def score_question_master(person, messages):
    texts = user_texts(person, messages)
    if not texts:
        return 0.0, "0.0% msgs end with '?'"
    cnt = sum([1 for t in texts if t.strip().endswith("?")])
    pct = (cnt / len(texts)) * 100.0
    sc = (pct / 25.0) if pct > 25.0 else (pct / 25.0) * 0.4
    return sc, f"{pct:.1f}% msgs end with '?'"


# Custom 9th archetype: THE EARLY BIRD (>50% messages between 05:00 and 09:59)
# Rare in college hostel chats where most activity happens late at night
def score_early_bird(idx, matrix):
    total = int(np.sum(matrix[idx]))
    if total == 0:
        return 0.0, "0.0% msgs between 05h-09h"
    morning = int(np.sum(matrix[idx, 5:10]))
    pct = (morning / total) * 100.0
    sc = (pct / 50.0) if pct > 50.0 else (pct / 50.0) * 0.4
    return sc, f"{pct:.1f}% msgs between 05h-09h"


def assign_archetypes(participants, messages, matrix, streaks, msg_count):
    all_scores = {}
    pool = []

    for idx, p in enumerate(participants):
        all_scores[p] = {
            "THE SPAMMER": score_spammer(p, messages),
            "THE GROUP MOM": score_group_mom(p, messages),
            "THE NIGHT OWL": score_night_owl(idx, matrix),
            "THE STORYTELLER": score_storyteller(p, messages),
            "THE DRAMA QUEEN": score_drama_queen(p, messages),
            "THE GHOST": score_ghost(p, streaks),
            "THE COMEDIAN": score_comedian(p, messages),
            "THE QUESTION MASTER": score_question_master(p, messages),
            "THE EARLY BIRD": score_early_bird(idx, matrix),
        }
        for arch, (sc, detail) in all_scores[p].items():
            pool.append((sc, msg_count[p], p, arch, detail))

    # Exclusive assignment: highest normalized score wins; ties broken by total messages
    pool.sort(key=lambda x: (x[0], x[1]), reverse=True)

    result = {}
    taken = set()
    for sc, _, p, arch, detail in pool:
        if p not in result and arch not in taken:
            result[p] = {"archetype": arch, "detail": detail}
            taken.add(arch)

    for p in participants:
        ranked = sorted(all_scores[p].items(), key=lambda x: x[1][0], reverse=True)
        for arch, _ in ranked:
            if arch != result[p]["archetype"]:
                result[p]["runner_up"] = arch
                break

    return result


if overview:
    archetypes = assign_archetypes(participants, messages, heatmap, streaks, overview["msg_count"])
    print("PERSONALITY ARCHETYPES")
    for p in participants:
        info = archetypes[p]
        print(f"  {p:<10} -> {info['archetype']:<18} ({info['detail']}) | Runner-up: {info['runner_up']}")


PERSONALITY ARCHETYPES
  Rahul      -> THE SPAMMER        (avg 4.5 msgs in a row) | Runner-up: THE NIGHT OWL
  Priya      -> THE GROUP MOM      (caring keyword score: 621 (64.5% msgs)) | Runner-up: THE QUESTION MASTER
  Neha       -> THE DRAMA QUEEN    (63.3% ALL-CAPS / dramatic msgs) | Runner-up: THE SPAMMER
  Aman       -> THE NIGHT OWL      (79.8% msgs between 23h-04h) | Runner-up: THE GROUP MOM
  Karan      -> THE STORYTELLER    (avg 57.0 words per msg) | Runner-up: THE GROUP MOM
  Vikas      -> THE GHOST          (silent on 44 of 60 days (73%)) | Runner-up: THE COMEDIAN


### Feature 8: Final Report


In [32]:
def print_report(group_name, messages, counts):
    w = 68
    if len(messages) == 0:
        print("=" * w)
        print("  GROUPDNA REPORT - NO MESSAGES FOUND")
        print("=" * w)
        return

    ov = get_overview(messages)
    parts = ov["participants"]
    pk = get_most_active(messages, ov["total_days"])
    hm = make_heatmap(messages, parts)
    top_w, _ = get_top_words(messages)
    _, fast, slow = get_response_stats(messages, parts)
    st = get_silent_streaks(messages, parts, ov["start_date"], ov["end_date"])
    archs = assign_archetypes(parts, messages, hm, st, ov["msg_count"])

    print("=" * w)
    print(f"  GROUPDNA REPORT  |  \"{group_name}\"")
    print(f"  {ov['total_messages']:,} messages  |  {len(parts)} members  |  {ov['total_days']} days")
    print("=" * w)

    hr = pk["busiest_hour"]
    print(f"  Period        : {ov['start_str']} to {ov['end_str']}")
    print(f"  Busiest day   : {pk['busiest_day']} ({pk['busiest_day_msgs']} messages)")
    print(f"  Busiest hour  : {hr:02d}:00 - {(hr + 1) % 24:02d}:00 (avg {pk['busiest_hour_avg']:.1f} msgs/day)")
    print(f"  Skipped/Meta  : {counts['system']} system | {counts['media']} media | {counts['deleted']} deleted")
    print("-" * w)

    print("  MESSAGES PER PERSON")
    max_m = ov["ranked"][0][1]
    for name, cnt in ov["ranked"]:
        pct = (cnt / ov["total_messages"]) * 100
        bar = "█" * max(1, int((cnt / max_m) * 20))
        print(f"  {name:<10} {bar:<20}  {cnt:>5} ({pct:>5.1f}%)")
    print("-" * w)

    print("  ACTIVITY HEATMAP (hour of day, 00 to 23)")
    for line in format_heatmap(hm, parts):
        print(line)
    print("-" * w)

    print("  THIS GROUP'S FAVOURITE WORDS")
    if top_w:
        max_wc = top_w[0][1]
        for word, cnt in top_w:
            bar = "█" * max(1, int((cnt / max_wc) * 26))
            print(f"  {word:<12} {bar:<26}  {cnt:>4}")
    print("-" * w)

    print("  RESPONSE PATTERNS")
    if fast and slow:
        print(f"  Fastest replier : {fast[0]:<10} (avg {fmt_time(fast[1])})")
        print(f"  Slowest replier : {slow[0]:<10} (avg {fmt_time(slow[1])})")
    else:
        print("  Single-member chat (response time N/A)")

    print("\n  LONGEST SILENT STREAKS")
    for name, info in sorted(st.items(), key=lambda x: x[1]["longest"], reverse=True):
        d = info["longest"]
        if d == 0:
            print(f"  {name:<10} : 0 days (never went silent)")
        elif d == 1:
            print(f"  {name:<10} : 1 day ({info['start'].strftime('%d %b')})")
        else:
            print(f"  {name:<10} : {d} days ({info['start'].strftime('%d %b')} - {info['end'].strftime('%d %b')})")
    print("-" * w)

    print("  PERSONALITY ARCHETYPES")
    for name in parts:
        info = archs[name]
        print(f"  {name:<10} -> {info['archetype']:<18} ({info['detail']})")
    print("=" * w)
    print("  Generated by GroupDNA  |  Python + NumPy")
    print("=" * w)


print_report(GROUP_NAME, messages, parse_counts)


  GROUPDNA REPORT  |  "Hostel Bois 4ever"
  3,174 messages  |  6 members  |  60 days
  Period        : 01 April 2024 to 30 May 2024
  Busiest day   : 04 May 2024 (76 messages)
  Busiest hour  : 18:00 - 19:00 (avg 4.1 msgs/day)
  Skipped/Meta  : 4 system | 32 media | 15 deleted
--------------------------------------------------------------------
  MESSAGES PER PERSON
  Rahul      ████████████████████    953 ( 30.0%)
  Priya      ███████████████         718 ( 22.6%)
  Neha       █████████████           635 ( 20.0%)
  Aman       ██████████              490 ( 15.4%)
  Karan      ███████                 354 ( 11.2%)
  Vikas      █                        24 (  0.8%)
--------------------------------------------------------------------
  ACTIVITY HEATMAP (hour of day, 00 to 23)
            00    03    06    09    12    15    18    21
  Rahul     . . . . . . . . . . . . ▒ ░ ░ ▒ ▒ ░ █ ▒ ░ █ ▒ ▒
  Priya     . . . . . . . ░ ▒ █ █ █ █ ▒ ▒ ░ ░ ▒ ▒ █ ▒ ░ ░ .
  Neha      . . . . . ░ . . ▒ █ █ ░ ▒ ▒ ░ 

### Edge Case Tests


In [33]:
# Test 1: Empty file
empty_m, empty_c = parse_chat([])
print_report("Empty Chat", empty_m, empty_c)

# Test 2: Single participant
solo_data = [
    "01/04/24, 08:00 - Parth created group \"Test\"",
    "01/04/24, 08:05 - Parth: bhai aaj assignment complete karna hai",
    "and push code to github",
    "01/04/24, 08:10 - Parth: <Media omitted>",
    "02/04/24, 09:00 - Parth: This message was deleted",
    "02/04/24, 09:15 - Parth: bhai scene done!",
]
solo_m, solo_c = parse_chat(solo_data)
print_report("Solo Chat", solo_m, solo_c)


  GROUPDNA REPORT - NO MESSAGES FOUND
  GROUPDNA REPORT  |  "Solo Chat"
  4 messages  |  1 members  |  2 days
  Period        : 01 April 2024 to 02 April 2024
  Busiest day   : 01 April 2024 (2 messages)
  Busiest hour  : 08:00 - 09:00 (avg 1.0 msgs/day)
  Skipped/Meta  : 1 system | 1 media | 1 deleted
--------------------------------------------------------------------
  MESSAGES PER PERSON
  Parth      ████████████████████      4 (100.0%)
--------------------------------------------------------------------
  ACTIVITY HEATMAP (hour of day, 00 to 23)
            00    03    06    09    12    15    18    21
  Parth     . . . . . . . . █ █ . . . . . . . . . . . . . .
  Legend: . (0-25%)  ░ (25-50%)  ▒ (50-75%)  █ (75-100%)
--------------------------------------------------------------------
  THIS GROUP'S FAVOURITE WORDS
  bhai         ██████████████████████████     2
  aaj          █████████████                  1
  assignment   █████████████                  1
  complete     ██████████

### Reflection
- **Hardest part:** Parsing raw WhatsApp text lines without regex while handling system lines, deleted messages, media stubs, and multi-line messages, and tuning exclusive archetype assignment so two people don't clash on the same label.
- **What I would do differently:** Add emoji frequency tracking and save the formatted final report directly to a `.txt` file alongside printing it.
